### Data Ingestion

In [5]:
import os

from langchain_core import documents
from langchain_core.documents import Document
import warnings
# Silence the migration notice specifically for your imports
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain_community")

from langchain_community.document_loaders import PyPDFLoader

In [6]:
loader = PyPDFLoader("../Documents/Bharatiya_Nagarik_Suraksha_Sanhita_2023.pdf")
pages = loader.load()

In [ ]:
print(pages)

### Embedding

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimensions: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"failed to load embedding model: {self.model_name} with error: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generating embeddings of texts length: {len(texts)}...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"embedding shape: {embeddings.shape}")
        return embeddings



embeddingManager = EmbeddingManager()
embeddingManager

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5150.93it/s]


model loaded successfully. Embedding dimensions: 384


In [20]:
import os
class VectorStore:
     def __init__(self, collection_name: str = "legal_documents", persist_directory: str = "../vector_store"):

         self.collection_name = collection_name
         self.persist_directory = persist_directory
         self.client = None
         self.collection = None
         self.initialize_store()

     def initialize_store(self):
         try:
             os.makedirs(self.persist_directory, exist_ok=True)
             self.client = chromadb.PersistentClient(path=self.persist_directory)
             self.collection = self.client.get_or_create_collection(name=self.collection_name,
                 metadata={ "description": "PDF document embeddings for RAG"}
             )
             print(f"initialized collection: {self.collection_name}")
             print(f"initialized persist directory: {self.collection.count()}")
         except Exception as e:
             print(f"failed to initialize store: {e}")
             raise e


     def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
         if len(documents) != len(embeddings):
             raise ValueError("Number of documents does not match the number of embeddings")

         print(f"adding documents: {len(documents)}...")

         ids=[]
         metadatas = []
         documents_text = []
         embedding_list = []

         for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

             doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
             ids.append(doc_id)

             metadata = dict(doc.metadata)
             metadata["doc_index"] = doc_id
             metadata["content_length"] = len(doc.page_content)

             metadatas.append(metadatas)

             documents_text.append(doc.page_content)
             embedding_list.append(embedding.tolist())

         try:
             self.collection.add(
                 ids=ids,
                 embeddings=embeddings,
                 metadatas=metadatas,
                 documents_text=documents_text
             )
             print(f"added document: {len(documents)} documents to vector store")
             print(f"total documents in collection: {self.collection.count()}")

         except Exception as e:
             print(f"failed to add document: {e}")
             raise e

vectorStore = VectorStore()
vectorStore

initialized collection: legal_documents
initialized persist directory: 0


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


Split 282 documents into 0 chunks


[]

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../Documents")

Found 3 PDF files to process

Processing: Bharatiya_Nagarik_Suraksha_Sanhita_2023.pdf
